In [36]:
class Value:
    def __init__(self, data, children=(), op = ""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(children)
        self._op = op

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
            other = other if isinstance(other, Value) else Value(other)
            out = Value(self.data + other.data, (self, other), "+")

            def _backward():
                self.grad += out.grad
                other.grad += out.grad

            out._backward = _backward
            return out
    
    def __mul__(self, other):
            other = other if isinstance(other, Value) else Value(other)
            out = Value(self.data * other.data, (self, other), "*")

            def _backward():
                self.grad += other.data * out.grad
                other.grad += self.data * out.grad

            out._backward = _backward
            return out    

    def relu(self):
            out = Value(max(0, self.data), (self,), "relu")

            def _backward():
                self.grad += (1.0 if out.data > 0 else 0.0) * out.grad

            out._backward = _backward
            return out

    def backward(self):
            topo = []
            visited = set()

            def build_topo(v):
                if v not in visited:
                    visited.add(v)
                    for child in v._prev:
                        build_topo(child)
                    topo.append(v)

            build_topo(self)
            self.grad = 1.0
            for v in reversed(topo):
                v._backward()

    def __pow__(self, n):
        out = Value(self.data ** n, (self,), f"**{n}")
        def _backward():
            self.grad += n * (self.data ** (n - 1)) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        import math
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")

        def _backward():
            self.grad += (1 - t ** 2) * out.grad

        out._backward = _backward
        return out

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __radd__(self, other):
        return self + other

x1 = Value(2.0)
x2 = Value(3.0)
y = (x1 * x2 + 1).relu()
y.backward()
print(y.data, x1.grad, x2.grad)


7.0 3.0 2.0


In [16]:
x = Value(2.0)
y = Value(3.0)
z = Value(4.0)
L = (x * y) + (x + z)
L.backward()
print(x.grad, y.grad, z.grad)
print(L._prev)
L.backward()
print(x.grad, y.grad, z.grad)
print(L._prev)

4.0 2.0 1.0
{Value(data=6.0, grad=1.0), Value(data=6.0, grad=1.0)}
12.0 6.0 3.0
{Value(data=6.0, grad=2.0), Value(data=6.0, grad=2.0)}


In [26]:
p = Value(2.0)
q = p ** 3
q.backward()
print(q.data, p.grad)

[Value(data=2.0, grad=0.0), Value(data=8.0, grad=1.0)]
8.0 12.0


In [28]:
u = Value(0.0)
v = u.tanh()
v.backward()
print(v.data, u.grad)

w = Value(2.0)
z = w.tanh()
z.backward()
print(round(z.data, 4), round(w.grad, 4))

[Value(data=0.0, grad=0.0), Value(data=0.0, grad=1.0)]
0.0 1.0
[Value(data=2.0, grad=0.0), Value(data=0.9640275800758169, grad=1.0)]
0.964 0.0707


In [31]:
w1, x1 = Value(2.0), Value(1.0)
w2, x2 = Value(1.0), Value(1.0)
b = Value(1.0)
y = (w1 * x1 + w2 * x2 + b).relu()
y.backward()
print(w1.grad, x1.grad, w2.grad, x2.grad, b.grad)

import torch
tw1 = torch.tensor(2.0, requires_grad=True)
tx1 = torch.tensor(1.0, requires_grad=True)
tw2 = torch.tensor(1.0, requires_grad=True)
tx2 = torch.tensor(1.0, requires_grad=True)
tb = torch.tensor(1.0, requires_grad=True)
ty = torch.relu(tw1 * tx1 + tw2 * tx2 + tb)
ty.backward()
print(tw1.grad.item(), tx1.grad.item(), tw2.grad.item(), tx2.grad.item(), tb.grad.item())

1.0 2.0 1.0 1.0 1.0
1.0 2.0 1.0 1.0 1.0


In [32]:
class Dual:
    def __init__(self, value, deriv):
        self.value = value
        self.deriv = deriv

    def __mul__(self, other):
        other = other if isinstance(other, Dual) else Dual(other, 0.0)
        return Dual(
            self.value * other.value,
            self.deriv * other.value + self.value * other.deriv,
        )

x = Dual(2.0, 1.0)
y = x * x * x
print(y.value, y.deriv)

8.0 12.0


In [33]:
def numerical_derivative(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

x0 = 2.0
numeric = numerical_derivative(lambda t: t ** 3, x0)
p = Value(x0)
q = p ** 3
q.backward()
auto = p.grad
rel = abs(numeric - auto) / max(abs(numeric), abs(auto), 1e-8)
print(f"numeric={numeric:.8f}  auto={auto:.8f}  rel={rel:.2e}")

numeric=12.00000000  auto=12.00000000  rel=1.77e-11


In [41]:
import random

class Neuron:
    def __init__(self, n_inputs):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(n_inputs)]
        self.b = Value(0.0)

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()

    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, n_inputs, n_outputs):
        self.neurons = [Neuron(n_inputs) for _ in range(n_outputs)]

    def __call__(self, x):
        return [n(x) for n in self.neurons]

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, sizes):
        self.layers = [Layer(sizes[i], sizes[i + 1]) for i in range(len(sizes) - 1)]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x[0] if len(x) == 1 else x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

random.seed(42)
model = MLP([2, 4, 1])
xs = [[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]]
ys = [-1.0, 1.0, 1.0, -1.0]

for step in range(100):
    preds = [model([Value(a), Value(b)]) for a, b in xs]
    loss = sum((p - y) ** 2 for p, y in zip(preds, ys))
    for p in model.parameters():
        p.grad = 0.0
    loss.backward()
    for p in model.parameters():
        p.data -= 0.05 * p.grad
    if step % 20 == 0 or step == 99:
        print(f"step {step:3d}  loss = {loss.data:.4f}")

for x, y in zip(xs, ys):
    pred = model([Value(x[0]), Value(x[1])])
    print(x, y, round(pred.data, 3))

step   0  loss = 4.1491
step  20  loss = 2.9166
step  40  loss = 1.4733
step  60  loss = 0.6011
step  80  loss = 0.2936
step  99  loss = 0.1783
[0.0, 0.0] -1.0 -0.853
[0.0, 1.0] 1.0 0.771
[1.0, 0.0] 1.0 0.801
[1.0, 1.0] -1.0 -0.753
